In [1]:
import pandas as pd

In [2]:
from main import ShortStraddleBot,DataFeed

In [3]:
import time
import logging
import threading
from datetime import datetime, timedelta, timezone
from collections import deque
from dataclasses import dataclass, field
from typing import Optional

from delta_rest_client import DeltaRestClient, OrderType, TimeInForce

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
API_KEY    = "Qq2vhYvusJUBRzTBb42C0B1C9bh0xR"
API_SECRET = "iVrq55jfT0o03KJ6LeeVT8CPPnUYnU3rW71ozJmKqj06mXcrBUR86hqR13x8"

# Pick one BASE_URL:
BASE_URL   = "https://cdn-ind.testnet.deltaex.org"      # Demo India production
# BASE_URL = "https://api.delta.exchange"            # Global production
# BASE_URL = "https://cdn-ind.testnet.deltaex.org"   # India testnet
# BASE_URL = "https://testnet-api.delta.exchange"    # Global testnet

QUANTITY             = 1          # contracts per leg
SL_PCT               = 0.25       # 25 % SL on each leg premium
EMA_PERIOD           = 5          # EMA period on straddle price
SLOPE_THRESH         = -1.5       # EMA slope entry threshold
MAX_REENTRY          = 10         # max trades per session
REENTRY_WAIT         = 5 * 60    # cooldown seconds after SL hit
STRIKE_STEP          = 500        # BTC ATM rounding step

SESSION_START        = (4, 00)    # (hour, minute) UTC
SESSION_END          = (23, 55)   # exit all before daily expiry at midnight UTC

CANDLE_POLL_INTERVAL = 60         # seconds between signal checks
SL_MONITOR_INTERVAL  = 15         # seconds between SL polls

In [4]:
bot = ShortStraddleBot()

10:21:12 | INFO     | =======================================================
10:21:12 | INFO     |   Delta Exchange Short Straddle Bot — starting up
10:21:12 | INFO     | =======================================================
10:21:12 | INFO     |   BASE_URL      : https://cdn-ind.testnet.deltaex.org
10:21:12 | INFO     |   QUANTITY      : 1 contract(s) per leg
10:21:12 | INFO     |   SL_PCT        : 25%
10:21:12 | INFO     |   EMA_PERIOD    : 5
10:21:12 | INFO     |   SLOPE_THRESH  : -1.5
10:21:12 | INFO     |   MAX_REENTRY   : 10
10:21:12 | INFO     |   REENTRY_WAIT  : 300s
10:21:12 | INFO     |   SESSION       : 04:00 – 23:55 UTC
10:21:12 | INFO     |   Console level : INFO  |  File level: DEBUG
10:21:12 | INFO     | =======================================================
10:21:12 | INFO     | [Bot] All components ready.


In [6]:
sdk_client = DeltaRestClient(BASE_URL,API_KEY,API_SECRET)

In [7]:
datafeed = DataFeed(sdk_client)

In [9]:
data = datafeed.get_option_chain('17-05-2026')

10:24:48 | INFO     | [DataFeed] Option chain [17-05-2026]: 60 contracts


In [11]:
pd.DataFrame(data)

,top_tag,ltp_change_24h,tags,tick_size,underlying_asset_symbol,price_band,product_id,open,symbol,time,...,oi_value_symbol,volume,oi,turnover,product_trading_status,contract_type,high,greeks,contract_value,mark_price
0,,0.0000,[New],0.500000000000000000,BTC,"{'lower_limit': '2491.2942', 'upper_limit': '4...",181774,4000.0,P-BTC-80800-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.002,0.0020,156.0856,operational,put_options,4000.0,None,0.001000000000000000,2620.4
1,,0.0000,[New],0.500000000000000000,BTC,"{'lower_limit': '1087.68800054', 'upper_limit'...",181718,0.5,P-BTC-79400-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.001,0.0010,77.8769,operational,put_options,0.5,None,0.001000000000000000,1220.40000099
2,,7662.8571,[New],0.500000000000000000,BTC,"{'lower_limit': '687.197482', 'upper_limit': '...",181720,52.5,P-BTC-79000-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.325,0.2520,25525.7578,operational,put_options,4075.5,None,0.001000000000000000,820.00660022
3,,600.0000,[New],0.500000000000000000,BTC,"{'lower_limit': '491.21563925', 'upper_limit':...",181721,100.0,P-BTC-78800-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.065,0.0350,5091.2430,operational,put_options,700.0,None,0.001000000000000000,620.18345856
4,,0.0000,[New],0.500000000000000000,BTC,"{'lower_limit': '292.95885175', 'upper_limit':...",181722,217.0,P-BTC-78600-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.010,0.0100,780.9690,operational,put_options,217.0,None,0.001000000000000000,422.57198326
5,,1283.8710,[New],0.500000000000000000,BTC,"{'lower_limit': '105.27860017', 'upper_limit':...",181723,15.5,P-BTC-78400-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.462,0.3600,36096.4297,operational,put_options,214.5,None,0.001000000000000000,239.3204958
6,,-93.8144,[New],0.500000000000000000,BTC,"{'lower_limit': '0.5', 'upper_limit': '2673.48...",181985,97.0,P-BTC-78200-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.619,0.6090,48434.6225,operational,put_options,4120.0,None,0.001000000000000000,99.0871082
7,,-99.9895,[New],0.500000000000000000,BTC,"{'lower_limit': '0.5', 'upper_limit': '2572.54...",181986,4757.0,P-BTC-78000-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.410,0.3100,32018.7484,operational,put_options,4757.0,None,0.001000000000000000,26.30074569
8,,11800.0000,[New],0.500000000000000000,BTC,"{'lower_limit': '0.5', 'upper_limit': '2472.39...",181987,0.5,P-BTC-77800-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.296,0.1510,23077.3424,operational,put_options,94.0,None,0.001000000000000000,3.94815203
9,,525100.0000,[New],0.500000000000000000,BTC,"{'lower_limit': '0.5', 'upper_limit': '2374.70...",181988,0.5,P-BTC-77600-170526,2026-05-16T04:53:49.383033159Z,...,BTC,0.130,0.0800,10142.7270,operational,put_options,2626.0,None,0.001000000000000000,0.5


In [30]:
datafeed.get_1min_candles('P-BTC-78000-170526',)

[0.5]

In [35]:
datafeed.get_1min_candles('C-BTC-78000-170526')

11:16:36 | WARNING  | [DataFeed] candles(C-BTC-78000-170526): 0 bars returned — symbol may be illiquid or too new


[]

In [ ]:
sdk_client.get_h